In [1]:
from pyspark.sql import functions as F, Window
import com.microsoft.spark.fabric

dates = spark.read.synapsesql("wh_gold.dbo.dim_date")
fx_usd_eur = spark.table("lh_silver.dbo.fx_usd_eur")

dim_fx = dates.join(fx_usd_eur, on="date", how="left")

first_last_date = fx_usd_eur.agg(F.min(F.col("date")), F.max(F.col("date"))).first()
first_date = first_last_date[0]
last_date = first_last_date[1]

dim_fx = dim_fx.filter(F.col("date").between(first_date, last_date))

window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
dim_fx = dim_fx.withColumn("usd_per_eur", F.last(F.col("usd_per_eur"), ignorenulls=True).over(window))

dim_fx.write.mode("overwrite").synapsesql("wh_gold.dbo.dim_fx")

StatementMeta(, 4793907d-1326-434e-8323-e2f005211b41, 3, Finished, Available, Finished, True)